# 02 · EDA: the six history tables

**Goal:** check that the tables link together correctly, see how many applicants each table covers,
and find the strange values we must clean. Tables are loaded one at a time (only the needed columns)
to keep memory low. Run all cells from top to bottom (takes about 1–2 minutes).

In [ ]:
%load_ext autoreload
%autoreload 2

import gc

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from creditrisk.config import PROJECT_ROOT, get_path, set_seed

SEED = set_seed()
INTERIM, FIG = get_path("interim"), get_path("figures")
sns.set_theme(style="whitegrid", context="notebook")
facts = {}

HISTORY = ["bureau", "bureau_balance", "previous_application",
           "installments_payments", "pos_cash_balance", "credit_card_balance"]


def load(name, columns=None):
    """Load one Parquet table, optionally only some columns."""
    return pd.read_parquet(INTERIM / f"{name}.parquet", columns=columns)


def save(fig, name):
    fig.savefig(FIG / name, dpi=150, bbox_inches="tight")
    print(f"saved reports/figures/{name}")


train = load("application_train", ["SK_ID_CURR", "TARGET"])
test_ids = load("application_test", ["SK_ID_CURR"])["SK_ID_CURR"]
print("train applicants:", len(train), "| test applicants:", len(test_ids))

## 1. Keys are unique and train/test don't overlap

In [ ]:
checks = {
    "application_train: SK_ID_CURR unique": train["SK_ID_CURR"].is_unique,
    "application_test: SK_ID_CURR unique": test_ids.is_unique,
    "bureau: SK_ID_BUREAU unique": load("bureau", ["SK_ID_BUREAU"])["SK_ID_BUREAU"].is_unique,
    "previous_application: SK_ID_PREV unique":
        load("previous_application", ["SK_ID_PREV"])["SK_ID_PREV"].is_unique,
    "no SK_ID_CURR in both train and test":
        np.intersect1d(train["SK_ID_CURR"], test_ids).size == 0,
}
for name, ok in checks.items():
    print(f"{'OK  ' if ok else 'FAIL'}  {name}")
facts["all_key_checks_passed"] = all(checks.values())

## 2. Coverage: how many applicants appear in each table?
`bureau_balance` has no `SK_ID_CURR`, so it is linked through `bureau` (`SK_ID_BUREAU` → `SK_ID_CURR`).

In [ ]:
ids_in = {}
for name in ["bureau", "previous_application", "installments_payments",
             "pos_cash_balance", "credit_card_balance"]:
    ids = load(name, ["SK_ID_CURR"])["SK_ID_CURR"]
    ids_in[name] = (ids.unique(), len(ids))

bureau_keys = load("bureau", ["SK_ID_CURR", "SK_ID_BUREAU"])
bb_bureau_ids = load("bureau_balance", ["SK_ID_BUREAU"])["SK_ID_BUREAU"].unique()
has_bb = bureau_keys["SK_ID_BUREAU"].isin(bb_bureau_ids)
ids_in["bureau_balance"] = (bureau_keys.loc[has_bb, "SK_ID_CURR"].unique(), None)
facts["%_bureau_loans_with_monthly_rows"] = round(100 * has_bb.mean(), 1)
facts["%_bureau_balance_loans_not_in_bureau"] = round(
    100 * (~pd.Series(bb_bureau_ids).isin(bureau_keys["SK_ID_BUREAU"])).mean(), 1)

rows = []
for name in HISTORY:
    unique_ids, n_rows = ids_in[name]
    has = train["SK_ID_CURR"].isin(unique_ids)
    rows.append({
        "table": name,
        "% train applicants covered": round(100 * has.mean(), 1),
        "default % with": round(100 * train.loc[has, "TARGET"].mean(), 2),
        "default % without": round(100 * train.loc[~has, "TARGET"].mean(), 2),
    })
coverage = pd.DataFrame(rows).set_index("table")
coverage.to_csv(PROJECT_ROOT / "reports" / "coverage.csv")
facts["coverage_%"] = coverage["% train applicants covered"].to_dict()
display(coverage)
print("saved reports/coverage.csv")
del bureau_keys
gc.collect()

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.barh(coverage.index[::-1], coverage["% train applicants covered"][::-1], color="#5B8DB8")
ax.set(title="Share of train applicants that appear in each table",
       xlabel="% of applicants", xlim=(0, 100))
save(fig, "fig10_table_coverage.png")

## 3. Missing values, DAYS_* ranges and the 365243 code, table by table
Look for impossible values (for example end dates more than 100 years ago) and for the fake 365243 code.

In [ ]:
for name in HISTORY:
    df = load(name)
    print(f"\n=== {name}: {df.shape[0]:,} rows x {df.shape[1]} cols ===")
    miss = (100 * df.isna().mean()).round(1)
    miss = miss[miss > 0].sort_values(ascending=False)
    if len(miss):
        display(miss.rename("missing %").to_frame())
    else:
        print("no missing values")
    day_cols = [c for c in df.columns if c.startswith(("DAYS_", "MONTHS_"))]
    if day_cols:
        ranges = df[day_cols].agg(["min", "max"]).T
        ranges["count of 365243"] = (df[day_cols] == 365243).sum()
        per_year = [12 if c.startswith("MONTHS_") else 365.25 for c in ranges.index]
        ranges["min in years"] = (ranges["min"] / per_year).round(1)
        display(ranges)
        has_code = ranges["count of 365243"] > 0
        facts[f"{name}: columns with 365243"] = list(ranges.index[has_code])
    del df
    gc.collect()

## 4. Status values

In [ ]:
status_columns = [
    ("bureau", "CREDIT_ACTIVE"),
    ("bureau_balance", "STATUS"),
    ("previous_application", "NAME_CONTRACT_STATUS"),
    ("pos_cash_balance", "NAME_CONTRACT_STATUS"),
]
for name, col in status_columns:
    vc = load(name, [col])[col].value_counts()
    vc = vc[vc > 0]
    table = pd.DataFrame({"rows": vc, "%": (100 * vc / vc.sum()).round(2)})
    print(f"\n{name}.{col}")
    display(table)
    facts[f"{name}.{col} %"] = table["%"].to_dict()

## Key facts
Copy these numbers into your Findings below.

In [ ]:
for key, value in facts.items():
    print(f"{key:<45} {value}")

## Findings  ✍️ (write 8–12 bullets, each with a number)

- All key checks passed: IDs are unique and no applicant is in both train and test.
- Coverage of train applicants: bureau __%, previous applications __%, installments __%, POS __%, credit cards __%, bureau_balance __%.
- Applicants with no bureau history default at __% vs __% with history.
- __% of bureau loans have monthly rows in bureau_balance.
- previous_application uses the fake 365243 code in: __.
- Impossible values found: __ (for example DAYS_CREDIT_ENDDATE as low as __ years).
- Most common statuses: bureau loans __% closed / __% active; previous applications __% approved / __% refused.

## Decisions  ✍️ (cleaning rules for the feature code)

- previous_application: 365243 → NaN in __.
- bureau: clip or set NaN for impossible `DAYS_*` values below __.
- After joining, count features → 0 for applicants with no rows; means and ratios stay NaN; add `has_*` flags.
- bureau_balance: summarise per bureau loan first, then per applicant (two steps).